## Imports

In [2]:
import sys
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import *
from tqdm import tqdm
from paramest_nn.quantum_tools import *
from qutip import *
from pathlib import Path
DATAPATH = Path("D:\Dev\ParamEst-NN\ParamEst-NN-main\data")

## Generation of training data for 1D case (estimation of $\Delta$)

Set up parameter to create the model of the system in Qutip

In [2]:
N=2
omega=1
gamma=1

- I will generate randomly the parameters, and only one trajectory per parameter, in order to avoid any weird information leaking.
- Number of clicks per trajectory: `njumpsMC`
- Total number of trajectories generated for training: `ndeltas`

In [3]:
# Number of clicks per trajectory 这里改序列长度
njumpsMC=40

# Number of total trajectories to generate 这里改轨迹总数
ndeltas = 40000

# Generation of a list of random values of Delta 这里改Δ的范围
deltamin = 0
deltamax = 5.
delta_rand_list = deltamin + (deltamax-deltamin)*rand(ndeltas)
omega_list = omega*np.ones(ndeltas)

param_rand_list = list(zip(delta_rand_list,omega_list))

In [4]:
from qutip.solver.parallel import loky_pmap

trajectories = np.asarray(
    loky_pmap(
        generate_clicks_TLS,
        param_rand_list,
        task_kwargs=dict(njumpsMC=njumpsMC, gamma=gamma),
        progress_bar="tqdm",
        map_kw={"num_cpus": 8}
    )
)


d:\Dev\Conda\envs\ParamEstNN\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 40000/40000 [05:45<00:00, 115.64it/s]


The trajectories created have the form of time delays

In [5]:
trajectories.shape

(40000, 40)

See example of single trajectory

In [6]:
trajectories[0]

array([ 2.94247661,  7.09175075,  4.905293  ,  7.03943201,  9.18356104,
        1.6602684 ,  6.15142113,  9.37499043,  5.56202282, 10.17156745,
       10.11966834,  5.24757288, 18.26239006,  1.35428883,  6.552487  ,
        3.53846022, 31.01539663, 10.83987467,  6.23028384,  2.45963466,
       14.5930026 ,  3.60666543,  9.91286319,  3.35923843,  4.68600375,
        9.86833546,  3.87508248,  6.03126108, 25.97892337, 17.86888061,
       15.74486738,  0.50764497,  1.82828869,  0.66220744,  6.13256933,
        0.76853957,  3.20944807,  1.69889496,  5.37438783,  0.32664807])

Save the trajetories into the `[datapath]/training-trajectories/1D-delta` folder

In [7]:
filedir = DATAPATH /'training-trajectories/1D-delta/'
create_directory(filedir)

filename = filedir/'taus-Delta-1D'
filenameDeltas = filedir/'delta_rand_list-Delta-1D'

np.save(filename,trajectories)
np.save(filenameDeltas,delta_rand_list)

Directory D:\Dev\ParamEst-NN\ParamEst-NN-main\data\training-trajectories\1D-delta not found: creating...


## Generation of training data for 2D case (estimation of $\Delta$ and $\Omega$)

We sample the parameters randomly, and create only one trajectory per parameter

In [6]:
nparams = 100000

deltamin = 0.
deltamax = 3.

omegamin =0.25
omegamax = 5.

omega_rand_list = omegamin + (omegamax-omegamin)*rand(nparams)
delta_rand_list = deltamin + (deltamax-deltamin)*rand(nparams)

param_rand_list = list(zip(delta_rand_list,omega_rand_list))
param_rand_array = np.array(param_rand_list)

#### 生成训练集

In [7]:
from qutip.solver.parallel import loky_pmap

trajectories = np.array(
    loky_pmap(
        generate_clicks_TLS,
        param_rand_list,
        map_kw={"num_cpus": 12},
        progress_bar="tqdm",
    )
)


d:\Dev\Conda\envs\ParamEstNN\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 100000/100000 [09:58<00:00, 166.97it/s]


##### Save the trajetories into the `data/training/2D-delta-omega` folder

In [8]:
# Save files to disk

filedir = DATAPATH / 'training-trajectories/2D-delta-omega/'
create_directory(filedir)

filename = filedir / 'taus2D'
filenameParams = filedir / 'param_rand_list-2D'

np.save(filename,trajectories)
np.save(filenameParams,param_rand_array)

Folder already exists.


## Generation of validation clicks for 1D case

Setup Qutip operators

In [13]:
N=2
a=destroy(N)
psi0=basis(N,0)
c_ops=[]
gamma=1
c_ops.append(np.sqrt(gamma)*a)


Notice that for the validation, we will generate many trajectories for the same value of the parameter, unlike in the training data.

- Number of points in the grid of $\Delta$ values considered for the validation: `ndelta`

- Number of trajectories for each value of the grid of parameters: `ntraj`



In [ ]:
ndelta = 40
ntraj = 100

omega=1
deltaMax = 2.1
t0=0

njumpsMC=40 
njumpsSim = 50 # to actually run, we choose much larger than the number of clicks we plan to keep

deltagrid = np.linspace(0.,deltaMax,ndelta)
jumpArray = np.zeros((ndelta,ntraj,njumpsMC))

In [15]:
for idx_delta in tqdm(range(ndelta)):
    delta = deltagrid[idx_delta]
    # Estimate the final time to have, on average, njumpsSim jumps
    tf=njumpsSim*(4*delta**2+8*omega**2+gamma**2)/(4*gamma*omega**2)
    H=delta*a.dag()*a+omega*a+omega*a.dag()
    tlist=[t0,tf] #

    ntrajRemaining = ntraj
    jumpList = []
    trajs_to_complete = [[] for i in range(ntraj)]
    trajs_completed   = []

    while ntrajRemaining > 0:
        mc=mcsolve(H,psi0,tlist,c_ops,[],ntrajRemaining,progress_bar=False)
        time_array = mc.col_times

        # Append the results into the trajectories in trajs_to_complete
        # WARNING: Appending different trajectories into a single one is only valid for the TLS, which gets reseted after each jump
        for idx, times in enumerate(time_array):
            trajs_to_complete[idx]+= list(to_time_delay(times))

        sizes = np.array([len(traj) for traj in trajs_to_complete])
        good_indexes = np.arange(ntrajRemaining)[sizes>=njumpsMC]
        bad_indexes = np.arange(ntrajRemaining)[sizes<njumpsMC]
        n_good_traj = len(good_indexes)
        ntrajRemaining = ntrajRemaining - n_good_traj

        # Trajectories with njumpsMC jumps are moved into trajs_completed
        for idx in good_indexes:
            trajs_completed += [trajs_to_complete[idx][:njumpsMC]]

        # Remove completed trajectory from trajs_to_complete
        trajs_to_complete = [trajs_to_complete[idx] for idx in bad_indexes]


    jumpArray[idx_delta,:,:] = np.asarray(trajs_completed)

  0%|          | 0/40 [00:00<?, ?it/s]d:\Dev\Conda\envs\ParamEstNN\lib\site-packages\qutip\solver\solver_base.py:501: FutureWarning: "progress_bar" is now included in options:
 Use `options={"progress_bar": False / True / "tqdm" / "enhanced"}`
  warnings.warn(
d:\Dev\Conda\envs\ParamEstNN\lib\site-packages\qutip\solver\solver_base.py:598: FutureWarning: e_ops will be keyword only from qutip 5.3 for all solver
  warnings.warn(
d:\Dev\Conda\envs\ParamEstNN\lib\site-packages\qutip\solver\solver_base.py:598: FutureWarning: ntraj will be keyword only from qutip 5.3 for all solver
  warnings.warn(
100%|██████████| 40/40 [00:27<00:00,  1.44it/s]


In [16]:
filedir = DATAPATH / 'validation-trajectories/1D-delta/'
create_directory(filedir)

filenameJumps = filedir / f'validation-trajectories-1D-delta-nsets-{ntraj}'
filenameParams = filedir / f'validation-deltas-1D-delta-nsets-{ntraj}'
np.save(filenameJumps,jumpArray)
np.save(filenameParams,deltagrid)

Directory D:\Dev\ParamEst-NN\ParamEst-NN-main\data\validation-trajectories\1D-delta not found: creating...


## Generation of validation clicks for 2D case

Setup Qutip operators

In [17]:
N=2
a=destroy(N)
psi0=basis(N,0)
c_ops=[]
gamma=1
c_ops.append(np.sqrt(gamma)*a)


Notice that for the validation, we will generate many trajectories for the same value of the parameter pairs, unlike in the training data.

- Number of points in the grid of $\Delta$ values considered for the validation: `ndelta`
- Number of points in the grid of $\Omega$ values considered for the validation: `nomega`
- Number of trajectories for each value of the square grid of parameters: `ntraj`



In [18]:
ndelta = 40
nomega = 40

ntraj = 100
deltamin = 0.
deltamax = 2.1

omegamin =0.25
omegamax = 2.1

njumpsMC=40 # to keep
njumpsSim = 50 # to actually run, we choose much larger than the number of clicks we plan to keep


deltagrid = np.linspace(deltamin,deltamax,ndelta)
omegagrid = np.linspace(omegamin,omegamax,nomega)

def gen_param_list(array1,array2):
    return np.array(np.meshgrid(array1,array2)).T.reshape(-1,2)

param_grid = gen_param_list(deltagrid,omegagrid)
nparams = ndelta*nomega

jumpArray = np.zeros((nparams,ntraj,njumpsMC))

In [19]:
for idx_param in tqdm(range(nparams)):
    delta = param_grid[idx_param][0]
    omega = param_grid[idx_param][1]
    tf=njumpsSim*(4*delta**2+8*omega**2+gamma**2)/(4*gamma*omega**2)
    H=delta*a.dag()*a+omega*a+omega*a.dag()
    tlist=[0.,tf] #

    ntrajRemaining = ntraj
    jumpList = []
    trajs_to_complete = [[] for i in range(ntraj)]
    trajs_completed   = []

    while ntrajRemaining > 0:
        mc=mcsolve(H,psi0,tlist,c_ops,[],ntrajRemaining,progress_bar=False)
        time_array = mc.col_times

        # Append the results into the trajectories in trajs_to_complete
        # WARNING: Appending different trajectories into a single one is only valid for the TLS, which gets reseted after each jump
        for idx, times in enumerate(time_array):
            trajs_to_complete[idx]+= list(to_time_delay(times))

        sizes = np.array([len(traj) for traj in trajs_to_complete])
        good_indexes = np.arange(ntrajRemaining)[sizes>=njumpsMC]
        bad_indexes = np.arange(ntrajRemaining)[sizes<njumpsMC]
        n_good_traj = len(good_indexes)
        ntrajRemaining = ntrajRemaining - n_good_traj

        # Trajectories with njumpsMC jumps are moved into trajs_completed
        for idx in good_indexes:
            trajs_completed += [trajs_to_complete[idx][:njumpsMC]]

        # Remove completed trajectory from trajs_to_complete
        trajs_to_complete = [trajs_to_complete[idx] for idx in bad_indexes]


    jumpArray[idx_param,:,:] = np.asarray(trajs_completed)

100%|██████████| 1600/1600 [21:57<00:00,  1.21it/s]


In [20]:
filedir = DATAPATH / 'validation-trajectories/2D-delta-omega/'
create_directory(filedir)

filenameJumps = filedir / f'validation-trajectories-2D-delta-omega-nsets-{ntraj}'
filenameParams = filedir / f'validation-deltas-2D-delta-omega-nsets-{ntraj}'
np.save(filenameJumps,jumpArray)
np.save(filenameParams,param_grid)

Directory D:\Dev\ParamEst-NN\ParamEst-NN-main\data\validation-trajectories\2D-delta-omega not found: creating...


Given the high volume of trajectories, subsequent analysis can benefit from a splitted dataset.

Here, we split and save the datasets in 10 batches for future parallelization.

In [21]:
nbatches = 10
ntraj_per_batch = int(ntraj/nbatches)

In [22]:
jumpArray2 = np.reshape(jumpArray,(1600,nbatches,ntraj_per_batch,40))

In [23]:
for i in range(nbatches):
    np.save(filedir / f"validation-trajectories-10ktrajs-batch-{i}.npy",jumpArray2[:,i,:,:])